# 071 — Sensores, series y percepción en el borde

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Ventanas deslizantes:** el flujo del sensor se corta en ventanas de `w` muestras con
salto `h`: `n = 1 + ⌊(T − w)/h⌋`. `w` debe cubrir el fenómeno (un paso ~1 s, una caída
~2 s); `h` fija la latencia de decisión. Con solapamiento hay más ejemplos — y riesgo de
fuga si se parte train/test por ventana en vez de **por sujeto o sesión**.

**Features temporales:** media, desviación, **RMS** (`√(Σx²/n)`), cruces por cero (ZCR),
energía por banda de la FFT. Un clasificador clásico sobre estas features es un baseline
duro en HAR con cómputo mínimo.

**Cuantización int8:** `x ≈ s·(q − z)`; simétrica: `z = 0`, `s = max|x|/127`. 4× menos
memoria y aritmética entera (clave en MCU sin FPU); caída típica ~1 punto
(post-entrenamiento), menos con QAT.

**TinyML:** inferencia en microcontroladores (64-256 kB RAM, mW) con runtimes como
TFLite Micro / LiteRT; *pruning*, *distillation* y *duty-cycling* (un detector diminuto
despierta al modelo grande). Motivos del borde: latencia, energía, autonomía y
**privacidad** — la señal cruda nunca sale del dispositivo.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** (a) `T = 60·25 = 1500`, `w = 4·25 = 100`, `h = 2·25 = 50` →
`n = 1 + (1500−100)/50 = 29` ventanas. (b) Con h = w: `n = 1 + 1400/100 = 15`. Solapar casi
duplica los ejemplos y evita perder eventos en el borde de ventana; el riesgo es la fuga
train/test si se parte por ventana: ventanas vecinas comparten el 50 % de las muestras.

**Ejercicio 2.** A: media 0, RMS 1, ZCR 7. B: media 0, RMS 1, ZCR 1. Media y RMS son
idénticas; solo la **ZCR** las separa: captura contenido frecuencial (A oscila rápido, B
lento) sin necesidad de FFT. Por eso un solo estadístico rara vez basta.

**Ejercicio 3.** (a) `s = 1.6/127 ≈ 0.01260`. (b) `q = round(0.5/0.01260) = round(39.7) =
40`; dequantizado `40·0.01260 ≈ 0.504`; error ≈ 0.004 (menor que medio paso de
cuantización, s/2 ≈ 0.0063). (c) float32: `120 000 · 4 = 480 kB` → **no** cabe en 256 kB;
int8: `120 000 · 1 = 120 kB` → sí cabe, con margen para el runtime (las activaciones van
aparte, en RAM).

**Ejercicio 4.** Implementación debajo: imprime 29, 15, las features y el error de
cuantización.


In [ ]:
result = run_lab("robotics", seed=71)
assert result["kind"] == "robotics"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicios 1 y 2 — verificados con código
def n_ventanas(T, w, h):
    return 1 + (T - w) // h

def features(x):
    n = len(x)
    media = sum(x) / n
    rms = (sum(v * v for v in x) / n) ** 0.5
    zcr = sum(1 for a, b in zip(x, x[1:]) if a * b < 0)
    return round(media, 3), round(rms, 3), zcr

print("ventanas (h=50):", n_ventanas(1500, 100, 50))
print("ventanas (h=w=100):", n_ventanas(1500, 100, 100))
print("A:", features([1, -1, 1, -1, 1, -1, 1, -1]))
print("B:", features([1, 1, 1, 1, -1, -1, -1, -1]))


In [ ]:
# Ejercicio 3 — cuantización int8 simétrica
def quantize(x, s):
    return max(-127, min(127, round(x / s)))

s = 1.6 / 127
q = quantize(0.5, s)
deq = q * s
print("s:", round(s, 5), "| q(0.5):", q, "| dequant:", round(deq, 4),
      "| error:", round(abs(0.5 - deq), 4))
print("float32:", 120_000 * 4 // 1000, "kB | int8:", 120_000 // 1000, "kB")


## Reflexión

1. Tu HAR da 97 % validando por ventanas al azar y 78 % validando por sujeto. ¿Cuál de los
   dos números reportas, y qué mecanismo exacto produce la brecha?
2. Un detector de caídas debe decidir en menos de 1 s, pero la batería debe durar meses.
   ¿Qué combinación de hop, duty-cycle y confirmación multi-ventana propones y qué
   trade-off acepta cada elección?
3. Para un sensor de audio doméstico, ¿qué ventaja concreta de privacidad da inferir en el
   borde, y qué telemetría podrías subir a la nube sin traicionarla?
